# DataSage Stage 2: Data Enrichment GRPO Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/training/train_enrichment.ipynb)

Trains **Qwen2.5-3B** via [Unsloth](https://github.com/unslothai/unsloth) + [TRL GRPO](https://huggingface.co/docs/trl/grpo_trainer) to enrich enterprise datasets with derived fields and lookups.

The model learns to add enrichment columns (salary bands, risk scores, velocity metrics, SLA flags) by interacting with the **DataSage Enrichment** OpenEnv environment.

**Runtime:** T4 GPU (Colab free tier) | ~30-45 min

---

## 1. Setup

In [ ]:
# Verify GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "GPU not detected! Go to Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)")

In [ ]:
%%capture
!pip install unsloth vllm
!pip install trl>=0.26 transformers accelerate peft bitsandbytes
!pip install wandb datasets requests

In [ ]:
# --- API Keys ---
# Option A: Use Colab Secrets (recommended) — add HF_TOKEN and WANDB_API_KEY in the key icon on the left sidebar
# Option B: Paste them below
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("Loaded keys from Colab Secrets")
except Exception:
    pass  # Fill manually below if not using Colab Secrets

# Uncomment and fill if not using Colab Secrets:
# os.environ["HF_TOKEN"] = "hf_..."
# os.environ["WANDB_API_KEY"] = "wandb_..."

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or above"
print("Keys loaded.")

## 2. Load Model (Unsloth 4-bit)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {total:,} total, {trainable:,} trainable ({100*trainable/total:.1f}%)")

## 3. Dataset

64 enrichment prompts across 4 enterprise domains (HR, Sales, Project Management, IT Operations).

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a data enrichment agent. You enrich enterprise datasets by adding "
    "derived fields, lookups, and computed columns across multiple domains "
    "(HR, Sales, Project Management, IT Operations).\n\n"
    "Each turn, analyze the schema and respond with a JSON enrichment action:\n"
    '{"operation": "<op>", "field_name": "<name>", "source": "<source>", '
    '"logic": "<logic>", "params": {}}\n\n'
    "Available operations:\n"
    "- add_field: Add a new enrichment field from a known source\n"
    "- lookup: Look up external reference data\n"
    "- compute_derived: Compute a derived metric from existing columns\n"
    "- add_category: Add a categorical classification\n\n"
    "Available enrichment sources per domain:\n"
    "- HR: salary_band, tenure_risk, satisfaction_index, industry_benchmark, flight_risk_score\n"
    "- Sales: deal_size_category, velocity_score, win_probability_model, industry_code, competitive_risk\n"
    "- PM: schedule_risk_score, resource_utilization, dependency_chain_depth, burndown_rate, delay_probability\n"
    "- IT Ops: sla_compliance_flag, mttr_band, escalation_path, incident_severity_score, recurring_pattern_flag\n\n"
    "Think step by step: examine the current schema, identify what enrichments "
    "would add the most value, then act."
)

TASK_PROMPTS = [
    # --- HR (16) ---
    "This HR dataset has MonthlyIncome but no salary band classification. Add salary_band.",
    "The HR data has YearsAtCompany but no risk assessment. Add tenure_risk.",
    "We need a satisfaction_index combining multiple satisfaction factors. Enrich the data.",
    "Add industry_benchmark salary data to compare compensation to market rates.",
    "Compute a flight_risk_score combining tenure, satisfaction, and overtime factors.",
    "The HR dataset needs salary bands for compensation analysis. Add the right enrichment.",
    "We want to identify employees at risk of leaving. Add the most relevant fields.",
    "Enrich HR data with all available sources. Start with the most impactful.",
    "The dataset has JobRole and MonthlyIncome. Add salary_band to categorize compensation.",
    "Management wants to understand attrition drivers. Add tenure_risk and satisfaction_index.",
    "The HR enrichment coverage is only 20%. Add fields to improve analytical value.",
    "We need to benchmark salaries against industry. Add industry_benchmark.",
    "Add a computed flight risk metric considering multiple employee factors.",
    "The HR schema is missing derived analytics. Add satisfaction_index as a composite metric.",
    "Enrich employee records with salary band classifications for compensation review.",
    "The HR data has raw metrics but no risk scores. Start enriching with tenure_risk.",
    # --- Sales (16) ---
    "The Sales pipeline has Amount data but no deal categorization. Add deal_size_category.",
    "We need deal velocity tracking. Add velocity_score based on DaysInStage.",
    "Add win_probability_model to predict deal outcomes from stage and velocity.",
    "The sales data needs industry classification codes. Add industry_code.",
    "Compute competitive_risk scores for deals at risk of being lost.",
    "The pipeline lacks predictive metrics. Add win_probability_model first.",
    "Enrich deals with size categories and velocity scores for pipeline analysis.",
    "The Sales enrichment coverage is 0%. Start adding the most valuable fields.",
    "We need to categorize deals for forecasting. Add deal_size_category based on Amount.",
    "Sales leadership wants velocity metrics. Add velocity_score.",
    "Add industry codes for market segmentation analysis.",
    "The pipeline needs risk scoring. Add competitive_risk.",
    "Enrich the sales dataset to support better forecasting. Start with win_probability_model.",
    "Add deal size categorization: Small/Medium/Large/Enterprise.",
    "The sales team needs velocity analytics. Enrich with velocity_score.",
    "Add all available sales enrichments. Start with the most impactful.",
    # --- Project Management (16) ---
    "The PM dataset has CompletionPct but no schedule risk assessment. Add schedule_risk_score.",
    "We need resource utilization metrics. Add resource_utilization.",
    "Add dependency_chain_depth to understand task blocking relationships.",
    "Compute burndown_rate to track project completion velocity.",
    "Add delay_probability to predict tasks likely to miss deadlines.",
    "The PM data needs risk scoring. Add schedule_risk_score first.",
    "Enrich project tasks with utilization and burndown metrics.",
    "The PM enrichment coverage is very low. Add fields for project analytics.",
    "We need to identify at-risk tasks. Add schedule_risk_score and delay_probability.",
    "Add resource utilization tracking to understand team capacity.",
    "The project data has completion percentages but no rate metrics. Add burndown_rate.",
    "Enrich tasks with dependency depth for critical path analysis.",
    "Project managers need delay predictions. Add delay_probability.",
    "Add schedule risk scoring for deadline management.",
    "The PM data has hours data but no utilization ratio. Add resource_utilization.",
    "Enrich project tasks with all PM enrichments. Start with schedule_risk_score.",
    # --- IT Operations (16) ---
    "The IT Ops data has SLA targets but no compliance flags. Add sla_compliance_flag.",
    "We need MTTR classification. Add mttr_band.",
    "Add escalation_path recommendations based on ticket category and priority.",
    "Compute incident_severity_score from priority and customer impact.",
    "Add recurring_pattern_flag to identify recurring issues.",
    "The IT Ops data needs SLA monitoring. Add sla_compliance_flag first.",
    "Enrich tickets with severity scoring and MTTR bands.",
    "The IT Ops enrichment coverage is 0%. Start adding enrichments.",
    "We need to classify resolution times. Add mttr_band.",
    "Add escalation path recommendations for incident management.",
    "The ticket data needs a severity score. Add incident_severity_score.",
    "Add recurring pattern detection to identify systemic issues.",
    "Enrich IT tickets with SLA compliance flags.",
    "The ops team needs MTTR analytics. Add mttr_band.",
    "Add all available IT Ops enrichments for incident analytics.",
    "We need to flag recurring incidents. Add recurring_pattern_flag.",
]

dataset = Dataset.from_dict({
    "prompt": [
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": p}]
        for p in TASK_PROMPTS
    ]
})

print(f"Dataset: {len(dataset)} prompts across 4 domains")

## 4. Reward Functions

Three reward signals:
1. **Environment reward** — steps through the deployed OpenEnv enrichment environment
2. **JSON format reward** — well-formed JSON with valid operation and source
3. **Reasoning reward** — chain-of-thought before acting

In [ ]:
import json
import re
import requests
import time

ENV_URL = "https://ricalanis-datasage-enrichment.hf.space"

VALID_OPS = {"add_field", "lookup", "compute_derived", "add_category"}
VALID_SOURCES = {
    "salary_band", "tenure_risk", "satisfaction_index", "industry_benchmark",
    "flight_risk_score", "deal_size_category", "velocity_score",
    "win_probability_model", "industry_code", "competitive_risk",
    "schedule_risk_score", "resource_utilization", "dependency_chain_depth",
    "burndown_rate", "delay_probability", "sla_compliance_flag", "mttr_band",
    "escalation_path", "incident_severity_score", "recurring_pattern_flag",
}

# --- Parser ---
def parse_enrichment_action(text):
    """Extract an enrichment action dict from model output."""
    m = re.search(r'\{[^{}]*"operation"[^{}]*\}', text, re.DOTALL)
    if m:
        try:
            data = json.loads(m.group())
            if "operation" in data:
                return data
        except json.JSONDecodeError:
            pass
    # Fallback
    t = text.lower()
    for source in VALID_SOURCES:
        if source.replace("_", " ") in t or source in t:
            return {"operation": "add_field", "field_name": source, "source": source, "logic": "", "params": {}}
    return {"operation": "add_field", "field_name": "unknown", "source": "", "logic": "", "params": {}}


# --- Reward 1: Environment ---
_env_calls = 0
_env_total_reward = 0.0

def env_reward_fn(completions, **kwargs):
    """Step through the enrichment environment. Primary reward signal."""
    global _env_calls, _env_total_reward
    rewards = []
    for text in completions:
        try:
            action = parse_enrichment_action(text)
            requests.post(f"{ENV_URL}/reset", json={}, timeout=10)
            r = requests.post(f"{ENV_URL}/step", json={"action": action}, timeout=10)
            reward = float(r.json().get("reward", 0.0))
        except Exception:
            reward = 0.0
        rewards.append(reward)
    _env_calls += 1
    _env_total_reward += sum(rewards) / len(rewards)
    if _env_calls % 5 == 0:
        print(f"  [env_reward] call {_env_calls}, running avg: {_env_total_reward/_env_calls:.3f}")
    return rewards


# --- Reward 2: JSON format ---
def json_format_reward(completions, **kwargs):
    """Reward well-formed JSON enrichment actions."""
    rewards = []
    for text in completions:
        m = re.search(r'\{[^{}]*"operation"[^{}]*\}', text)
        if m:
            try:
                data = json.loads(m.group())
                op_ok = data.get("operation") in VALID_OPS
                field_ok = "field_name" in data and data["field_name"] != "unknown"
                source_ok = data.get("source", "") in VALID_SOURCES
                if op_ok and field_ok and source_ok:
                    rewards.append(1.0)
                elif op_ok and field_ok:
                    rewards.append(0.6)
                elif op_ok:
                    rewards.append(0.3)
                else:
                    rewards.append(0.2)
            except (json.JSONDecodeError, AttributeError):
                rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards


# --- Reward 3: Reasoning ---
def reasoning_reward(completions, **kwargs):
    """Reward chain-of-thought before the enrichment action."""
    rewards = []
    for text in completions:
        score = 0.0
        t = text.lower()
        if any(w in t for w in ["first", "let me", "i should", "step 1", "think", "analyze"]):
            score += 0.3
        if any(w in t for w in ["enrich", "add", "derive", "compute", "coverage", "value"]):
            score += 0.2
        rewards.append(min(score, 0.5))
    return rewards


# Quick env sanity check
try:
    r = requests.get(f"{ENV_URL}/health", timeout=10)
    print(f"Environment health: {r.status_code}")
except Exception as e:
    print(f"WARNING: Environment not reachable ({e}). Training will use fallback rewards.")

## 5. Train with GRPO

In [ ]:
from trl import GRPOConfig, GRPOTrainer

os.environ["WANDB_PROJECT"] = "datasage"

training_args = GRPOConfig(
    output_dir="./outputs/enrichment-grpo",
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=256,
    max_prompt_length=512,
    logging_steps=1,
    save_steps=50,
    bf16=True,
    use_vllm=True,
    vllm_mode="colocate",
    report_to="wandb",
    run_name="datasage-enrichment-grpo",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[
        env_reward_fn,       # Primary: environment reward
        json_format_reward,  # Auxiliary: structured output
        reasoning_reward,    # Auxiliary: chain-of-thought
    ],
)

print(f"Starting GRPO training...")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Generations per prompt: {training_args.num_generations}")
print(f"  Reward functions: env, json_format, reasoning")
print()

t_start = time.time()
trainer.train()
elapsed = (time.time() - t_start) / 60
print(f"\nTraining complete in {elapsed:.1f} minutes")
print(f"Environment reward calls: {_env_calls}, avg reward: {_env_total_reward/_env_calls:.3f}" if _env_calls else "")

## 6. Save & Push to Hub

In [ ]:
HF_MODEL_REPO = "ricalanis/datasage-qwen-enrichment"

trainer.save_model("./outputs/enrichment-grpo-final")
tokenizer.save_pretrained("./outputs/enrichment-grpo-final")
print("Model saved locally.")

trainer.push_to_hub(HF_MODEL_REPO)
print(f"Pushed to https://huggingface.co/{HF_MODEL_REPO}")